# EXP-012 판정 근거: 문장 경계 청킹(512 + 256)

판정은 보류(Recall@5 +0.13%p, 95% 구간 [−0.23, +0.51])다. 판정과 관계없이 순위가 바뀐 질의마다 dense·BM25·조건 목록 중 어디서 움직였는지, 512·256 중 어느 청크가 정답 문서 점수를 냈는지, 정답 청크 길이가 어떻게 달라졌는지 보고, 운영 측정(청크 수·용량, 질의당 검색 시간, 콜드 스타트)을 기준(EXP-011)과 같은 PC에서 잰다. 구간은 SPEC과 같은 회의 단위 paired bootstrap(`src/rag/eval/bootstrap.py`, `configs/eval/spec_v1.yaml`)이다.

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from rag.conditions import build_extractor
from rag.eval.bootstrap import paired_bootstrap
from rag.eval.metrics import gold_ranks, load_cfg, load_gold, min_docs, per_query, read_run
from rag.index import load_cfg as load_run_cfg
from rag.retrieve import Retriever, chunks_by_doc, conditions_first
from rag.search import query_embeddings, query_files, query_list, read_texts

C:\workspace\meet-rag-exp012\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
INDEX = ROOT / 'data' / 'index'
BASE = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-dev-full.csv'
CAND = 'snowflake-ko-sentence-512-64-hybrid-cond-filter-law-ms256-dev-full.csv'
CONFIGS = {'base': ['+exp=exp011', 'index.scope=full'], 'cand': ['+exp=exp012', 'index.scope=full']}
INDEX_DIRS = {'base': ['snowflake-ko-fixed-512-64', 'snowflake-ko-fixed-256-32'],
              'cand': ['snowflake-ko-sentence-512-64', 'snowflake-ko-sentence-256-32']}

## 지표 차이와 95% 구간

In [3]:
cfg = load_cfg()
ev, bs = cfg.eval, cfg.eval.bootstrap
gold = load_gold(cfg, 'dev-full')
n_min = min_docs(ev.recall_ks, ev.mrr_k)
scores = {name: per_query(gold_ranks(read_run(RUNS / f), gold, n_min), ev.recall_ks, ev.mrr_k)
          for name, f in [('base', BASE), ('cand', CAND)]}
rows = []
for m in scores['base'].columns:
    d = scores['cand'][m] - scores['base'][m]
    point, lo, hi = paired_bootstrap(d.to_numpy(), gold[bs.unit].to_numpy(), bs.n_resamples,
                                     bs.seed, bs.alpha)
    rows.append({'metric': m, 'base': scores['base'][m].mean(), 'cand': scores['cand'][m].mean(),
                 'diff_%p': point * 100, 'lo_%p': lo * 100, 'hi_%p': hi * 100})
print(len(gold), 'queries')
print(pd.DataFrame(rows).set_index('metric').round(4))

3016 queries
             base    cand  diff_%p   lo_%p   hi_%p
metric                                            
recall@1   0.8369  0.8329  -0.3979 -1.2056  0.4137
recall@5   0.9337  0.9350   0.1326 -0.2297  0.5057
recall@10  0.9572  0.9569  -0.0332 -0.4119  0.3381
mrr@10     0.8815  0.8798  -0.1624 -0.6293  0.3038


In [4]:
rb = gold_ranks(read_run(RUNS / BASE), gold, n_min).fillna(99)
rc = gold_ranks(read_run(RUNS / CAND), gold, n_min).fillna(99)
for k in (1, 5):
    up, down = int(((rb > k) & (rc <= k)).sum()), int(((rb <= k) & (rc > k)).sum())
    print(f'top{k}: 진입 {up}, 이탈 {down}')
print('정답 순위가 바뀐 질의', int((rb != rc).sum()))

top1: 진입 63, 이탈 75
top5: 진입 19, 이탈 15
정답 순위가 바뀐 질의 296


## 청크 구성

문장 경계 청크가 고정 청크보다 몇 개 많은지, 청크 길이(토큰)가 어떻게 달라졌는지 본다. 문장 경계는 상한 안에서 문장을 다 넣지 못하면 청크를 끝내므로 평균 길이가 짧아진다.

In [5]:
def du(p):
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def read_chunks(name):
    with (INDEX / name / 'full' / 'chunks.jsonl').open(encoding='utf-8') as f:
        return pd.DataFrame([{k: r[k] for k in ('doc_id', 'n_tokens')} for r in map(json.loads, f)])


chunks = {name: read_chunks(name) for names in INDEX_DIRS.values() for name in names}
rows = []
for name, c in chunks.items():
    d = INDEX / name / 'full'
    rows.append({'index': name, 'chunks': len(c), 'tokens_mean': c['n_tokens'].mean(),
                 'tokens_p50': c['n_tokens'].median(), 'tokens_sum_M': c['n_tokens'].sum() / 1e6,
                 'emb_MiB': du(d / 'emb') / 2**20,
                 'chunks_jsonl_MiB': (d / 'chunks.jsonl').stat().st_size / 2**20})
sizes = pd.DataFrame(rows).set_index('index')
print(sizes.round(1))
for k, (b, c) in {'512': INDEX_DIRS['base'][0::2] + INDEX_DIRS['cand'][0::2],
                  '256': INDEX_DIRS['base'][1::2] + INDEX_DIRS['cand'][1::2]}.items():
    print(k, f"청크 수 {sizes.loc[c, 'chunks'] / sizes.loc[b, 'chunks']:.3f}배")

                              chunks  tokens_mean  tokens_p50  tokens_sum_M  \
index                                                                         
snowflake-ko-fixed-512-64     118039        435.6       512.0          51.4   
snowflake-ko-fixed-256-32     220645        236.4       256.0          52.2   
snowflake-ko-sentence-512-64  118228        421.0       484.0          49.8   
snowflake-ko-sentence-256-32  223983        218.6       236.0          49.0   

                              emb_MiB  chunks_jsonl_MiB  
index                                                    
snowflake-ko-fixed-512-64       461.1             252.6  
snowflake-ko-fixed-256-32       861.9             266.6  
snowflake-ko-sentence-512-64    461.8             244.9  
snowflake-ko-sentence-256-32    874.9             252.0  
512 청크 수 1.002배
256 청크 수 1.015배


## 단계별 순위와 운영 측정

기준·후보 검색기를 차례로 읽어(콜드 스타트), 질의를 하나씩 넣어 정답 문서의 목록별 순위(dense·BM25·조건 목록), 최종 순위, 정답 문서 dense 점수를 낸 청크(512·256 중 어디, 토큰 수)를 기록하고 질의당 시간(질의 임베딩 제외, 점수 계산부터 최종 순위까지)을 잰다. 운영처럼 질의를 하나씩 받는다고 보고 행렬곱도 질의마다 한다. 최종 순위가 순위 파일과 같은지 확인한다.

In [6]:
def run_config(key):
    run_cfg = load_run_cfg(CONFIGS[key])
    t0 = time.perf_counter()
    ret = Retriever(run_cfg)
    load_sec = time.perf_counter() - t0
    qids, text_path = query_list(run_cfg)
    q_emb = query_embeddings(run_cfg, qids, text_path, query_files(run_cfg)['emb'])
    texts = read_texts(text_path, qids)
    ex = build_extractor(run_cfg)
    conds = {q: ex.extract(t) for q, t in zip(qids, texts, strict=True)}
    tokens = ret.query_tokens(texts)
    n512 = len(chunks[INDEX_DIRS[key][0]])
    n_tok = np.concatenate([chunks[n]['n_tokens'].to_numpy() for n in INDEX_DIRS[key]])
    doc_chunks = chunks_by_doc(ret.dense_ids)
    rows, secs = [], []
    for j, qid in enumerate(qids):
        t = time.perf_counter()
        row = ret.chunk_emb @ q_emb[j]
        named, matched, _ = ret.lists(row, tokens[j], conds[qid])
        score = ret._rrf_scores(named)
        docs = conditions_first(sorted(score, key=score.__getitem__, reverse=True),
                                matched)[: ret.r.top_k]
        secs.append(time.perf_counter() - t)
        g = gold.loc[qid, 'doc_id']
        best = int(doc_chunks[g][np.argmax(row[doc_chunks[g]])])
        rec = {'qid': qid, 'final': docs.index(g) + 1 if g in docs else None,
               'size': '512' if best < n512 else '256', 'gold_tokens': int(n_tok[best]),
               'best_match': g in matched}
        for n, lst in named.items():
            rec[n] = lst.index(g) + 1 if g in lst else None
        rows.append(rec)
    del ret
    return load_sec, np.array(secs), pd.DataFrame(rows).set_index('qid')


stage, timing = {}, {}
for key in CONFIGS:
    load_sec, secs, stage[key] = run_config(key)
    timing[key] = {'load_s': load_sec, 'p50_ms': np.median(secs) * 1e3,
                   'p95_ms': np.percentile(secs, 95) * 1e3, 'max_ms': secs.max() * 1e3}
    print(key, stage[key].shape, {k: round(v, 1) for k, v in timing[key].items()})

base (3016, 7) {'load_s': 35.1, 'p50_ms': np.float64(88.4), 'p95_ms': np.float64(137.2), 'max_ms': np.float64(156.0)}
cand (3016, 7) {'load_s': 35.4, 'p50_ms': np.float64(91.7), 'p95_ms': np.float64(150.4), 'max_ms': np.float64(416.2)}


In [7]:
for key, r in [('base', rb), ('cand', rc)]:
    final = stage[key]['final'].fillna(99)
    print(key, '순위 파일과 최종 순위 일치', bool((final == r.loc[final.index]).all()))
print(stage['cand'].head(3))

base 순위 파일과 최종 순위 일치 True
cand 순위 파일과 최종 순위 일치 True
             final size  gold_tokens  best_match  dense  bm25  conditions
qid                                                                      
020437-0001    1.0  256          198       False    1.0   1.0         NaN
020473-0001    1.0  256          240       False    1.0   1.0         NaN
020483-0001    1.0  512          284       False    1.0   1.0         NaN


## 순위가 바뀐 질의의 단계별 이동

top5 진입·이탈 질의와 정답 순위가 바뀐 질의 전체에서 정답 문서의 dense·BM25 순위가 기준 → 후보로 오른 질의와 내린 질의 수를 센다(100위 밖은 101로 본다). 조건 목록은 청크와 무관하게 문서 단위로 만들지만 순서에 dense 문서 점수를 쓰므로 함께 본다.

In [8]:
b, c = (stage[k].reindex(gold.index) for k in ('base', 'cand'))
groups = {
    'top5 진입': (rb > 5) & (rc <= 5),
    'top5 이탈': (rb <= 5) & (rc > 5),
    'top1 진입': (rb > 1) & (rc == 1),
    'top1 이탈': (rb == 1) & (rc > 1),
    '순위 바뀜 전체': rb != rc,
}
rows = []
for gname, m in groups.items():
    rec = {'group': gname, 'n': int(m.sum())}
    for col in ('dense', 'bm25', 'conditions'):
        x, y = b.loc[m, col].fillna(101), c.loc[m, col].fillna(101)
        rec[f'{col} 오름'] = int((y < x).sum())
        rec[f'{col} 내림'] = int((y > x).sum())
    rows.append(rec)
print(pd.DataFrame(rows).set_index('group'))

            n  dense 오름  dense 내림  bm25 오름  bm25 내림  conditions 오름  \
group                                                                
top5 진입    19        15         2        6        3              0   
top5 이탈    15         1        11        3        7              0   
top1 진입    63        45         2       23        3              0   
top1 이탈    75         1        53        3       15              0   
순위 바뀜 전체  296       104       124       78       63              0   

          conditions 내림  
group                    
top5 진입               0  
top5 이탈               0  
top1 진입               0  
top1 이탈               0  
순위 바뀜 전체              0  


In [9]:
for col in ('dense', 'bm25'):
    x, y = b[col].fillna(101), c[col].fillna(101)
    print(f'{col} 전체: 정답 순위 오름 {int((y < x).sum())}, 내림 {int((y > x).sum())}, '
          f'top5 비율 {(x <= 5).mean():.4f} -> {(y <= 5).mean():.4f}')

dense 전체: 정답 순위 오름 245, 내림 272, top5 비율 0.9092 -> 0.9118
bm25 전체: 정답 순위 오름 193, 내림 127, top5 비율 0.9174 -> 0.9168


## 512·256 청크의 기여

정답 문서의 dense 점수(청크 최댓값)를 낸 청크가 512인지 256인지를 기준·후보에서 센다. 후보의 이동 질의에서 256 몫이 전체와 다르면 그 크기의 경계 변화가 이동을 이끈 것으로 읽는다.

In [10]:
rows = []
for gname, m in {'전체': rb > 0, **groups}.items():
    rows.append({'group': gname, 'n': int(m.sum()),
                 '기준 256 비율': (b.loc[m, 'size'] == '256').mean(),
                 '후보 256 비율': (c.loc[m, 'size'] == '256').mean()})
print(pd.DataFrame(rows).set_index('group').round(3))

             n  기준 256 비율  후보 256 비율
group                               
전체        3016      0.653      0.636
top5 진입     19      0.737      0.789
top5 이탈     15      0.800      0.600
top1 진입     63      0.698      0.762
top1 이탈     75      0.773      0.653
순위 바뀜 전체   296      0.676      0.649


## 정답 청크 길이

정답 문서 점수를 낸 청크의 토큰 수를 기준·후보에서 비교한다. 문장 경계로 바뀌며 짧아진 청크가 점수를 냈는지 본다.

In [11]:
rows = []
for gname, m in {'전체': rb > 0, **groups}.items():
    rows.append({'group': gname, 'n': int(m.sum()),
                 '기준 토큰 중앙값': b.loc[m, 'gold_tokens'].median(),
                 '후보 토큰 중앙값': c.loc[m, 'gold_tokens'].median(),
                 '후보가 짧음 비율': (c.loc[m, 'gold_tokens'] < b.loc[m, 'gold_tokens']).mean()})
print(pd.DataFrame(rows).set_index('group').round(3))

             n  기준 토큰 중앙값  후보 토큰 중앙값  후보가 짧음 비율
group                                          
전체        3016      256.0      243.0      0.659
top5 진입     19      256.0      247.0      0.789
top5 이탈     15      256.0      247.0      0.800
top1 진입     63      256.0      242.0      0.794
top1 이탈     75      256.0      244.0      0.680
순위 바뀜 전체   296      256.0      245.0      0.706


## 운영 측정

콜드 스타트는 검색기 생성(dense 임베딩·BM25 인덱스·조건 색인 읽기) 시간, 질의당 시간은 질의 임베딩을 뺀 검색 시간이다. 같은 PC에서 한 번씩 잰 값이라 다른 작업의 영향이 섞일 수 있다.

In [12]:
print(pd.DataFrame(timing).T.round(1))

      load_s  p50_ms  p95_ms  max_ms
base    35.1    88.4   137.2   156.0
cand    35.4    91.7   150.4   416.2
